# Lab 5: Transfer Learning - Cats vs Dogs Classification

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand transfer learning and why it works
- Use a pre-trained CNN (Convolutional Neural Network) as feature extractor
- Add a custom classification head for binary classification
- Work with image datasets from Kaggle
- Implement data augmentation for better generalization
- Evaluate model performance with appropriate metrics

**What You'll Build:** A cats vs dogs classifier achieving >90% accuracy using transfer learning

**Reference:** [Kaggle Cats vs Dogs CNN](https://www.kaggle.com/code/sachinpatil1280/cats-vs-dogs-image-classification-using-cnn-95/notebook)

**Note:** PyTorch and torchvision are pre-installed in Google Colab!

## Interactive Visualization: 2D Convolution

Understanding how convolutional neural networks "see" images starts with understanding convolution.

**Click 'Step' to see the kernel slide across the input and compute each output value!**

In [ ]:
%%html
<style>
#conv-container * { box-sizing: border-box; margin: 0; padding: 0; }
#conv-container { padding: 1rem 0; font-family: sans-serif; font-size: 13px; color: #1a1a1a; }
#conv-container h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
#conv-container .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
#conv-container .row { display: flex; align-items: flex-start; gap: 24px; flex-wrap: wrap; justify-content: center; }
#conv-container .section-label { font-size: 11px; font-weight: 500; color: #666; text-align: center; margin-bottom: 6px; letter-spacing: .04em; text-transform: uppercase; }
#conv-container .grid-wrap { display: inline-block; }
#conv-container .grid { display: grid; gap: 3px; }
#conv-container .cell {
  width: 44px; height: 44px; border-radius: 6px; border: 0.5px solid #ccc;
  display: flex; align-items: center; justify-content: center; font-size: 13px; font-weight: 500;
  color: #1a1a1a; background: #fff; transition: background .2s, border-color .2s, color .2s;
}
#conv-container .cell.hi-input { background: #E6F1FB; border-color: #185FA5; color: #0C447C; }
#conv-container .cell.hi-kernel { background: #EEEDFE; border-color: #534AB7; color: #3C3489; }
#conv-container .cell.hi-out { background: #FAEEDA; border-color: #854F0B; color: #633806; }
#conv-container .cell.done { background: #f7f7f5; color: #888; }
#conv-container .cell.active-out { background: #FAEEDA; border-color: #854F0B; color: #633806; }
#conv-container .op-area {
  margin-top: 16px; padding: 14px 16px; border-radius: 10px; border: 0.5px solid #ddd;
  background: #f7f7f5; font-size: 13px; line-height: 1.8; min-height: 72px;
}
#conv-container .term-r { color: #185FA5; font-weight: 500; }
#conv-container .term-p { color: #534AB7; font-weight: 500; }
#conv-container .term-o { color: #854F0B; font-weight: 500; }
#conv-container .sym { color: #888; }
#conv-container .controls { display: flex; flex-wrap: wrap; gap: 12px; margin-top: 14px; align-items: center; }
#conv-container .ctrl-group { display: flex; align-items: center; gap: 8px; font-size: 12px; color: #666; }
#conv-container select {
  font-size: 12px; padding: 4px 8px; border-radius: 6px; border: 0.5px solid #ccc;
  background: #fff; color: #1a1a1a;
}
#conv-container .btn {
  padding: 5px 14px; border-radius: 6px; border: 0.5px solid #ccc; background: transparent;
  color: #1a1a1a; font-size: 12px; cursor: pointer;
}
#conv-container .btn:hover { background: #f0f0ee; }
#conv-container .btn.primary { border-color: #185FA5; color: #185FA5; }
#conv-container .btn.primary:hover { background: #E6F1FB; }
#conv-container .btn:disabled { opacity: 0.5; cursor: not-allowed; }
#conv-container .step-row { display: flex; gap: 8px; margin-top: 14px; align-items: center; }
#conv-container .prog { flex: 1; height: 4px; border-radius: 2px; background: #ddd; overflow: hidden; }
#conv-container .prog-bar { height: 100%; background: #185FA5; border-radius: 2px; transition: width .3s; }
#conv-container .legend { display: flex; gap: 16px; flex-wrap: wrap; margin-top: 12px; font-size: 11px; color: #666; }
#conv-container .leg { display: flex; align-items: center; gap: 5px; }
#conv-container .leg-dot { width: 12px; height: 12px; border-radius: 3px; flex-shrink: 0; }
</style>

<div id="conv-container">
  <h1>2D Convolution — How CNNs "See" Images</h1>
  <p class="subtitle">Step through the sliding kernel to see how each output value is computed.</p>

  <div class="row">
    <div class="grid-wrap">
      <div class="section-label">Input (5×5)</div>
      <div class="grid" style="grid-template-columns:repeat(5,44px)" id="conv-gridInput"></div>
    </div>
    <div style="display:flex;flex-direction:column;align-items:center;justify-content:center;padding-top:28px;gap:6px">
      <div style="font-size:18px;color:#888">*</div>
      <div style="font-size:10px;color:#aaa">convolve</div>
    </div>
    <div class="grid-wrap">
      <div class="section-label">Kernel (3×3)</div>
      <div class="grid" style="grid-template-columns:repeat(3,44px)" id="conv-gridKernel"></div>
    </div>
    <div style="display:flex;flex-direction:column;align-items:center;justify-content:center;padding-top:28px;gap:6px">
      <div style="font-size:18px;color:#888">=</div>
    </div>
    <div class="grid-wrap">
      <div class="section-label">Feature map (3×3)</div>
      <div class="grid" style="grid-template-columns:repeat(3,44px)" id="conv-gridOutput"></div>
    </div>
  </div>

  <div class="op-area" id="conv-opArea">
    <span style="color:#888">Press <b>Step</b> to slide the kernel and compute one output cell at a time.</span>
  </div>

  <div class="step-row">
    <button class="btn" onclick="convReset()">↺ Reset</button>
    <button class="btn primary" onclick="convStepFwd()" id="conv-btnStep">Step →</button>
    <div class="prog"><div class="prog-bar" id="conv-progBar" style="width:0%"></div></div>
    <span style="font-size:11px;color:#666;min-width:40px;text-align:right" id="conv-stepLabel">0 / 9</span>
  </div>

  <div class="controls">
    <div class="ctrl-group">
      <span>Kernel preset:</span>
      <select id="conv-kernelSelect" onchange="convApplyPreset()">
        <option value="edge">Edge detector</option>
        <option value="blur">Blur (average)</option>
        <option value="sharpen">Sharpen</option>
        <option value="identity">Identity</option>
      </select>
    </div>
  </div>

  <div class="legend">
    <div class="leg"><div class="leg-dot" style="background:#E6F1FB;border:1px solid #185FA5"></div> Input patch (active)</div>
    <div class="leg"><div class="leg-dot" style="background:#EEEDFE;border:1px solid #534AB7"></div> Kernel weights</div>
    <div class="leg"><div class="leg-dot" style="background:#FAEEDA;border:1px solid #854F0B"></div> Output value</div>
  </div>
</div>

<script>
(function() {
  const INPUT = [
    [1,2,3,0,1],
    [0,1,2,3,1],
    [2,1,0,1,2],
    [1,3,2,1,0],
    [0,1,1,2,3]
  ];

  const PRESETS = {
    edge:     [[-1,-1,-1],[-1,8,-1],[-1,-1,-1]],
    blur:     [[1,1,1],[1,1,1],[1,1,1]],
    sharpen:  [[0,-1,0],[-1,5,-1],[0,-1,0]],
    identity: [[0,0,0],[0,1,0],[0,0,0]]
  };

  let kernel = PRESETS.edge.map(r=>[...r]);
  let step = 0;
  const TOTAL = 9;

  function getPos(s){ return [Math.floor(s/3), s%3]; }

  function computeOutput(ri, ci){
    let sum = 0;
    for(let kr=0;kr<3;kr++) for(let kc=0;kc<3;kc++) sum += INPUT[ri+kr][ci+kc]*kernel[kr][kc];
    return sum;
  }

  function allOutputs(){
    const O=[];
    for(let r=0;r<3;r++){O.push([]); for(let c=0;c<3;c++) O[r].push(computeOutput(r,c));}
    return O;
  }

  function renderInput(hiR, hiC){
    const el=document.getElementById('conv-gridInput');
    if (!el) return;
    el.innerHTML='';
    for(let r=0;r<5;r++) for(let c=0;c<5;c++){
      const d=document.createElement('div');
      d.className='cell';
      d.textContent=INPUT[r][c];
      if(hiR!==null && r>=hiR && r<hiR+3 && c>=hiC && c<hiC+3) d.classList.add('hi-input');
      el.appendChild(d);
    }
  }

  function renderKernel(highlight){
    const el=document.getElementById('conv-gridKernel');
    if (!el) return;
    el.innerHTML='';
    for(let r=0;r<3;r++) for(let c=0;c<3;c++){
      const d=document.createElement('div');
      d.className='cell';
      d.textContent=kernel[r][c];
      if(highlight) d.classList.add('hi-kernel');
      el.appendChild(d);
    }
  }

  function renderOutput(upTo, activeR, activeC){
    const el=document.getElementById('conv-gridOutput');
    if (!el) return;
    el.innerHTML='';
    const O=allOutputs();
    for(let r=0;r<3;r++) for(let c=0;c<3;c++){
      const d=document.createElement('div');
      d.className='cell';
      const idx=r*3+c;
      if(idx<upTo){ d.textContent=O[r][c]; d.classList.add('done'); }
      else if(r===activeR && c===activeC){ d.textContent=O[r][c]; d.classList.add('active-out'); }
      else{ d.textContent='?'; }
      el.appendChild(d);
    }
  }

  function buildOpHTML(ri, ci){
    const pairs=[];
    for(let kr=0;kr<3;kr++) for(let kc=0;kc<3;kc++){
      const iv=INPUT[ri+kr][ci+kc], kv=kernel[kr][kc];
      pairs.push(`<span class="term-r">${iv}</span><span class="sym">×</span><span class="term-p">${kv}</span>`);
    }
    const sum=computeOutput(ri,ci);
    const nums=[];
    for(let kr=0;kr<3;kr++) for(let kc=0;kc<3;kc++) nums.push(INPUT[ri+kr][ci+kc]*kernel[kr][kc]);
    return `
      <div><b>Output[${ri}][${ci}]</b> — element-wise multiply patch × kernel, then sum</div>
      <div style="margin-top:6px;line-height:2">${pairs.join('<span class="sym"> + </span>')}</div>
      <div style="margin-top:6px">= <span style="color:#888">${nums.join(' + ')}</span> = <span class="term-o" style="font-size:15px">${sum}</span></div>`;
  }

  function render(){
    if(step===0){
      renderInput(null,null);
      renderKernel(false);
      renderOutput(-1,null,null);
      const opArea = document.getElementById('conv-opArea');
      if (opArea) opArea.innerHTML='<span style="color:#888">Press <b>Step</b> to slide the kernel and compute one output cell at a time.</span>';
    } else {
      const [ri,ci]=getPos(step-1);
      renderInput(ri,ci);
      renderKernel(true);
      renderOutput(step-1,ri,ci);
      const opArea = document.getElementById('conv-opArea');
      if (opArea) opArea.innerHTML=buildOpHTML(ri,ci);
    }
    const pct=Math.round(step/TOTAL*100);
    const progBar = document.getElementById('conv-progBar');
    if (progBar) progBar.style.width=pct+'%';
    const stepLabel = document.getElementById('conv-stepLabel');
    if (stepLabel) stepLabel.textContent=step+' / '+TOTAL;
    const btnStep = document.getElementById('conv-btnStep');
    if (btnStep) {
      btnStep.disabled=(step>=TOTAL);
      btnStep.textContent=step>=TOTAL ? 'Done!' : 'Step →';
    }
  }

  window.convStepFwd = function(){ if(step<TOTAL){step++; render();} };
  window.convReset = function(){ step=0; render(); };

  window.convApplyPreset = function(){
    const sel = document.getElementById('conv-kernelSelect');
    if (!sel) return;
    const v=sel.value;
    kernel=PRESETS[v].map(r=>[...r]);
    step=0; render();
  };

  render();
})();
</script>


### Key Concepts Explained

Before we start coding, let's understand the key concepts for this lab:

#### 1. Convolutional Neural Networks (CNNs)

**What are CNNs?**
- Specialized neural networks for images
- Use **convolutional layers** that detect patterns (edges, textures, shapes)
- Learn hierarchical features: edges → textures → parts → objects

**Why CNNs for images?**
- Preserve spatial structure (unlike flattening to 1D)
- Parameter sharing (same filter applied everywhere)
- Translation invariant (detect cat anywhere in image)

#### 2. ReLU Activation Function

**Definition:** `ReLU(x) = max(0, x)`
- If x > 0: output = x
- If x ≤ 0: output = 0

**Why ReLU?**
- Introduces non-linearity (without it, deep networks = linear model)
- Computationally efficient
- Helps with gradient flow
- Most common activation in modern networks

#### 3. Dropout

**What is Dropout?**
- Randomly set some neurons to 0 during training
- Typical rate: 0.5 (drop 50%)
- At test time: use all neurons

**Why it prevents overfitting:**
- Forces network to not rely on any single neuron
- Creates ensemble effect
- Learns more robust features

#### 4. Sigmoid and BCEWithLogitsLoss

**Sigmoid:** Outputs probability between 0 and 1

**BCEWithLogitsLoss:** Combines sigmoid + binary cross entropy
- More numerically stable
- Standard for binary classification
- Model outputs raw logits (no sigmoid needed)

#### 5. Data Augmentation

**Random transformations** applied to training images:
- Horizontal flip, rotation, crop, color jitter
- Helps model generalize
- Only for training, not validation!

#### 6. ImageNet Normalization

Pre-trained models expect:
- Mean=[0.485, 0.456, 0.406]
- Std=[0.229, 0.224, 0.225]
- These are ImageNet RGB statistics

### Visualizing ReLU and Dropout

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

print("=" * 60)
print("Demo 1: ReLU Activation")
print("=" * 60)

# Create sample data
x = torch.linspace(-3, 3, 100)
y_relu = torch.relu(x)

plt.figure(figsize=(10, 4))

# Plot ReLU
plt.subplot(1, 2, 1)
plt.plot(x.numpy(), y_relu.numpy(), linewidth=2)
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Input')
plt.ylabel('Output')
plt.title('ReLU(x) = max(0, x)')
plt.grid(True, alpha=0.3)

# Show effect on sample values
sample_inputs = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
sample_outputs = torch.relu(sample_inputs)
plt.subplot(1, 2, 2)
plt.stem(sample_inputs.numpy(), sample_outputs.numpy())
plt.xlabel('Input')
plt.ylabel('ReLU Output')
plt.title('ReLU on Sample Values')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f\"Sample: {sample_inputs.tolist()} → {sample_outputs.tolist()}\")\nprint(\"Notice: Negative values become 0, positive values unchanged\")\n\nprint(\"\\n\" + \"=\"*60)\nprint(\"Demo 2: Dropout\")\nprint(\"=\"*60)\n\n# Demo dropout\ndropout = nn.Dropout(0.5)\nx = torch.ones(10)\n\nprint(\"Original:\", x.tolist())\n\n# Training mode (dropout active)\ndropout.train()\ny_train = dropout(x)\nprint(\"After Dropout (train mode):\", y_train.tolist())\nprint(\"Notice: ~50% are set to 0, rest are scaled up (×2)\")\n\n# Eval mode (no dropout)\ndropout.eval()\ny_eval = dropout(x)\nprint(\"After Dropout (eval mode):\", y_eval.tolist())\nprint(\"Notice: No dropout in eval mode - all values kept\")\n\nprint(\"\\n✓ ReLU and Dropout demonstrated!\")

**Setup:** Check GPU availability (Colab provides free T4 GPU!)

In [ ]:
import torch

# Detect device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU detected! Training will be fast.")
else:
    print("\n⚠️  No GPU detected. Training will be slower.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU")

## Part 1: What is Transfer Learning?

### The Big Idea

Instead of training a CNN from scratch (which requires millions of images and days of training), we:
1. **Use a pre-trained model** (trained on ImageNet - 14M images, 1000 classes)
2. **Freeze the feature extractor** (keep learned patterns: edges, textures, shapes)
3. **Add a new classification head** (train only this part for our specific task)

### Why Transfer Learning Works

**Low-level features** (edges, corners, colors) are universal:
- A ResNet trained on ImageNet learned to detect edges, textures, shapes
- These same features are useful for cats vs dogs!
- No need to relearn them

**Benefits:**
- ✅ **Faster training** (minutes vs days)
- ✅ **Less data needed** (thousands vs millions)
- ✅ **Better accuracy** (pre-trained features are high quality)
- ✅ **Less compute** (fine-tune on CPU or small GPU)

### Architecture

```
Input Image (3×224×224)
        ↓
Pre-trained ResNet18 (frozen) ← Trained on ImageNet
        ↓
Feature Vector (512 dimensions)
        ↓
Custom Head (trainable) ← We train this!
  - FC Layer 1: 512 → 128
  - ReLU
  - Dropout
  - FC Layer 2: 128 → 1
  - Sigmoid
        ↓
Output (0 = cat, 1 = dog)
```

## Part 2: Dataset Setup

### Kaggle Cats vs Dogs Dataset

**Dataset:** [Dogs vs Cats on Kaggle](https://www.kaggle.com/c/dogs-vs-cats/data)
- 25,000 labeled images
- 12,500 cats, 12,500 dogs
- Various sizes and orientations

### Downloading the Dataset

**Option 1: Direct Kaggle API (Recommended)**

In [ ]:
# Install kaggle package
!pip install -q kaggle

# Upload your kaggle.json (API token) to Colab
# Get it from: https://www.kaggle.com/settings → API → Create New API Token
# Then upload to Colab files panel

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle setup complete!")
print("Note: If kaggle.json not found, upload it from Kaggle settings.")

In [ ]:
# Method 2: Upload via code prompt (optional)
from google.colab import files

print("Click 'Choose Files' and select your kaggle.json")
uploaded = files.upload()

# Verify file was uploaded
if 'kaggle.json' in uploaded:
    print("\n✓ kaggle.json uploaded successfully!")
else:
    print("\n⚠️ kaggle.json not found. Please upload it.")

### Getting Started with Kaggle API in Colab

**Reference:** [Kaggle API in Colab Guide](https://colab.research.google.com/github/corrieann/kaggle/blob/master/kaggle_api_in_colab.ipynb)

#### Complete Setup Steps

1. **Create Kaggle account** at kaggle.com (free)
2. **Generate API token** at https://www.kaggle.com/settings → API → Create New API Token
3. **Upload kaggle.json** to Colab (drag & drop to files panel)
4. **Accept competition rules** at https://www.kaggle.com/c/dogs-vs-cats (click "I Understand and Accept")
5. Run the cell below to configure credentials

#### Step 4: Setup Kaggle Credentials

In [ ]:
# Setup Kaggle API (run after uploading your kaggle.json)

# Install kaggle package
!pip install -q kaggle

# Move kaggle.json to correct location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle API setup complete!")
print("\nVerifying credentials...")
!kaggle competitions list --page 1 | head -3

print("\n✓ Ready to download datasets!")

#### Step 5: Download Dataset

In [ ]:
# Download Dogs vs Cats dataset
# This downloads ~800MB, may take 2-3 minutes

print("Downloading dataset... (this may take 2-3 minutes)")
!kaggle competitions download -c dogs-vs-cats

print("\n✓ Download complete!")
print("\nExtracting files...")

# Unzip the main archive
!unzip -q dogs-vs-cats.zip

# Unzip the training images
!unzip -q train.zip

print("✓ Extraction complete!")
print(f"✓ Dataset ready: 25,000 images extracted")

# Check what we have
!ls -lh train/ | head -10

#### Troubleshooting

**Error: "401 Unauthorized"**
- Your kaggle.json is invalid
- Re-download from https://www.kaggle.com/settings

**Error: "403 Forbidden"**  
- Accept competition rules at https://www.kaggle.com/c/dogs-vs-cats
- Click "I Understand and Accept"

**Alternative: Manual Download**
- Download from Kaggle website
- Upload train.zip to Colab
- Run: `!unzip -q train.zip`

---

## Part 3: Data Loading and Preprocessing

### Image Preprocessing Requirements

Pre-trained models expect specific input:
1. **Size:** 224×224 pixels
2. **Normalization:** Mean=[0.485, 0.456, 0.406], Std=[0.229, 0.224, 0.225] (ImageNet stats)
3. **Format:** Tensor with shape (batch, 3, 224, 224)

### Data Augmentation

To improve generalization, we randomly transform training images:
- Random horizontal flip
- Random rotation (±10 degrees)
- Random crop and resize
- Color jitter (brightness, contrast)

**API Reference:**
- [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)
- [ImageFolder dataset](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data transformations
# Hint: Training needs augmentation, validation doesn't

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder('data/train', transform=train_transform)
val_dataset = datasets.ImageFolder('data/val', transform=val_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"Class to index: {train_dataset.class_to_idx}")

### Create Data Loaders

Data loaders handle batching and shuffling:

**Batch Size Considerations:**
- Too large: Out of memory
- Too small: Slow training, noisy gradients
- Typical: 32-64 for images

**API Reference:** [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

In [ ]:
# Create data loaders
# Hint: Use batch_size=32, shuffle=True for training, shuffle=False for validation

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test loading a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}  (batch_size, channels, height, width)")
print(f"Labels shape: {labels.shape}")
print(f"Sample labels: {labels[:5].tolist()}  (0=cat, 1=dog)")

### Visualize Augmented Images

In [ ]:
# Visualize what augmentation does
import numpy as np

def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Get a batch
images, labels = next(iter(train_loader))

# Show first 4 images
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)  # Clip to valid range
    axes[i].imshow(img)
    label_name = 'Cat' if labels[i] == 0 else 'Dog'
    axes[i].set_title(label_name, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Augmented Training Images', fontweight='bold')
plt.show()

## Exercise 1: Build Transfer Learning Model

### Your Task

Build a model using transfer learning:
1. Load a pre-trained ResNet18
2. Freeze all parameters (we don't want to retrain the feature extractor)
3. Replace the final layer with a custom classification head

### Architecture Hints

**Pre-trained Model:**
- Use `torchvision.models.resnet18(pretrained=True)`
- Original has 1000-class output (ImageNet)
- We need binary output (cat vs dog)

**Classification Head:**
```
Input: 512 features (from ResNet18)
  ↓
Linear(512 → 128)
  ↓
ReLU
  ↓
Dropout(0.5)  ← Prevents overfitting
  ↓
Linear(128 → 1)
  ↓
Sigmoid  ← Output between 0 and 1
```

**Loss Function:** BCEWithLogitsLoss (combines sigmoid + binary cross entropy)

**API Reference:**
- [torchvision.models.resnet18](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- [BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

### Starter Code

In [ ]:
import torch.nn as nn
import torchvision.models as models

class CatDogClassifier(nn.Module):
    """Transfer learning model for binary classification."""
    
    def __init__(self):
        super().__init__()
        
        # Load pre-trained ResNet18
        self.backbone = models.resnet18(pretrained=True)
        # Or in newer PyTorch: models.resnet18(weights='DEFAULT')
        
        # Freeze backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Replace final FC layer with custom classification head
        self.backbone.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)  # Binary output
        )
    
    def forward(self, x):
        return self.backbone(x)

### Test Your Model

In [ ]:
# Create model
model = CatDogClassifier()

# Check architecture
print("Model Architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (only classification head)")
print(f"Frozen parameters: {total_params - trainable_params:,}  (ResNet18 backbone)")

# Test forward pass
test_input = torch.randn(2, 3, 224, 224)  # Batch of 2 images
test_output = model(test_input)
print(f"\nInput shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}  (should be [2, 1])")
assert test_output.shape == (2, 1), f"Expected shape (2, 1), got {test_output.shape}"
print("\n✓ Model works!")

## Exercise 2: Implement Training Loop

### Training Setup

**Loss Function:** `BCEWithLogitsLoss`
- Combines sigmoid + binary cross-entropy
- More numerically stable than separate sigmoid + BCELoss
- Expects raw logits (no sigmoid in model)

**Optimizer:** Adam
- Learning rate: 0.001 (typical for transfer learning)
- Only optimize trainable parameters

**Training Loop Pattern:**
```python
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        # Forward
        outputs = model(images)
        loss = criterion(outputs.squeeze(), labels.float())
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### Your Task

Implement the complete training loop with:
- Loss tracking
- Accuracy calculation
- Progress printing

### Starter Code

In [ ]:
import torch.optim as optim

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = CatDogClassifier().to(device)

# Define loss function
criterion = nn.BCEWithLogitsLoss()

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training settings
num_epochs = 10

# Track metrics
train_losses = []
train_accs = []

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels.float())
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).long()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Acc: {epoch_acc*100:.2f}%")

print("\n✓ Training complete!")

### Expected Training Progress

With transfer learning, you should see:
```
Epoch [1/10], Loss: 0.3521, Acc: 85.23%
Epoch [2/10], Loss: 0.2134, Acc: 91.45%
Epoch [3/10], Loss: 0.1523, Acc: 94.12%
...
Epoch [10/10], Loss: 0.0823, Acc: 97.34%
```

**Note:** Training is fast because we only train the classification head!

## Exercise 3: Implement Validation

### Evaluation Metrics for Binary Classification

**Accuracy:** Overall correctness
- `accuracy = (TP + TN) / (TP + TN + FP + FN)`

**Precision:** Of predicted positives, how many are actually positive?
- `precision = TP / (TP + FP)`

**Recall:** Of actual positives, how many did we find?
- `recall = TP / (TP + FN)`

**F1 Score:** Harmonic mean of precision and recall
- `F1 = 2 * (precision * recall) / (precision + recall)`

Where:
- TP = True Positives (predicted dog, actually dog)
- TN = True Negatives (predicted cat, actually cat)
- FP = False Positives (predicted dog, actually cat)
- FN = False Negatives (predicted cat, actually dog)

### Your Task

Implement validation with multiple metrics:

**API Reference:** [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Solution: Validation with metrics

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images).squeeze()
        
        # Get predictions
        predictions = (torch.sigmoid(outputs) > 0.5).long()
        
        # Collect all predictions and labels
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"\nValidation Results:")
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")

### Target Performance

A well-trained model should achieve:
- **Accuracy:** > 90%
- **Precision:** > 88%
- **Recall:** > 88%
- **F1 Score:** > 89%

If metrics are lower:
- Train for more epochs
- Try different learning rate (0.0001 or 0.01)
- Add more data augmentation
- Try larger classification head

## Part 4: Visualizing Results

### Training Curves

In [ ]:
# Solution: Plot Training Curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss
ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Plot accuracy
ax2.plot([acc*100 for acc in train_accs], label='Train')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
# Solution: Confusion Matrix

import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cat', 'Dog'], 
            yticklabels=['Cat', 'Dog'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print(f"\nTrue Negatives (Cat→Cat): {cm[0,0]}")
print(f"False Positives (Cat→Dog): {cm[0,1]}")
print(f"False Negatives (Dog→Cat): {cm[1,0]}")
print(f"True Positives (Dog→Dog): {cm[1,1]}")

### Visualize Predictions

In [ ]:
# Solution: Visualize Predictions

# Get a batch from validation set
model.eval()
images, labels = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images).squeeze()
    predictions = (torch.sigmoid(outputs) > 0.5).long()

# Show first 8 predictions
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i].cpu()).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    
    true_label = 'Cat' if labels[i] == 0 else 'Dog'
    pred_label = 'Cat' if predictions[i].cpu() == 0 else 'Dog'
    correct = predictions[i].cpu() == labels[i]
    
    color = 'green' if correct else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Exercise 4: Analyze Model Performance

### Understanding Predictions

Let's look at:
1. **Confidence scores** (how certain is the model?)
2. **Misclassifications** (which images are hardest?)
3. **Per-class performance** (better at cats or dogs?)

### Your Task

Implement confidence analysis:

In [ ]:
# Solution: Get predictions with confidence scores

model.eval()
all_probs = []  # Confidence scores
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        predictions = (probs > 0.5).long()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("✓ Predictions collected with confidence scores!")

### Confidence Distribution

In [ ]:
# Solution: Confidence Distribution

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(all_probs[all_labels==0], bins=30, alpha=0.7, label='Cats', color='blue')
plt.hist(all_probs[all_labels==1], bins=30, alpha=0.7, label='Dogs', color='red')
plt.xlabel('Predicted Probability (Dog)')
plt.ylabel('Count')
plt.title('Confidence Distribution')
plt.legend()
plt.axvline(0.5, color='black', linestyle='--', label='Decision Boundary')

# Show correct vs incorrect
plt.subplot(1, 2, 2)
correct_mask = (all_preds == all_labels)
plt.hist(all_probs[correct_mask], bins=30, alpha=0.7, label='Correct', color='green')
plt.hist(all_probs[~correct_mask], bins=30, alpha=0.7, label='Incorrect', color='red')
plt.xlabel('Predicted Probability (Dog)')
plt.ylabel('Count')
plt.title('Correct vs Incorrect Predictions')
plt.legend()

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Peaks near 0 and 1 = confident predictions")
print("- Values near 0.5 = uncertain predictions")
print("- Incorrect predictions often have lower confidence")

### Find Hardest Examples

In [ ]:
# Uncomment after getting predictions:

# # Find misclassified images
# incorrect_mask = (all_preds != all_labels)
# incorrect_indices = np.where(incorrect_mask)[0]

# if len(incorrect_indices) > 0:
#     print(f"Found {len(incorrect_indices)} misclassifications")
#     print(f"Error rate: {len(incorrect_indices)/len(all_labels)*100:.2f}%")
#     
#     # Show 4 hardest mistakes
#     # Hardest = most confident but wrong
#     confidence = np.abs(all_probs - 0.5)  # Distance from decision boundary
#     incorrect_confidence = confidence[incorrect_mask]
#     hardest_indices = incorrect_indices[np.argsort(incorrect_confidence)[-4:]]
#     
#     # TODO: Visualize these hardest mistakes
#     # Hint: Load images from val_dataset using hardest_indices
#     # Show true label vs predicted label with confidence score
# else:
#     print("Perfect validation! No mistakes.")

## Bonus: Fine-Tuning

### What is Fine-Tuning?

After training the classification head, we can **unfreeze** some backbone layers and train them with a **very small learning rate**.

**Why?**
- Adapt pre-trained features to our specific task
- Can improve accuracy by 1-3%

**Risks:**
- Can overfit if not careful
- Takes longer to train
- Need to use smaller learning rate (0.0001)

### Try Fine-Tuning (Optional)

In [ ]:
# Uncomment to try fine-tuning:

# # Unfreeze last few layers of ResNet
# for param in model.backbone.layer4.parameters():
#     param.requires_grad = True

# # Create new optimizer with small learning rate
# optimizer_ft = optim.Adam([
#     {'params': model.backbone.layer4.parameters(), 'lr': 0.0001},  # Backbone: very small LR
#     {'params': model.backbone.fc.parameters(), 'lr': 0.001}         # Head: normal LR
# ])

# # Train for a few more epochs
# # (Copy training loop from Exercise 2, but use optimizer_ft)

print("Fine-tuning is optional - your model should already work well!")

## Summary

### What You've Learned

✅ **Transfer learning:** Leverage pre-trained models for new tasks

✅ **Feature extraction:** Use frozen backbone as feature extractor

✅ **Custom heads:** Add task-specific layers on top

✅ **Image preprocessing:** Resize, normalize, augment

✅ **Data loaders:** Efficient batching and loading

✅ **Binary classification:** BCEWithLogitsLoss for binary tasks

✅ **Evaluation metrics:** Accuracy, precision, recall, F1 score

✅ **Real dataset:** Kaggle Dogs vs Cats (25k images)

### Journey Complete: From Scratch to Production

**Lab 1:** Built arrays from pure Python → Understood NumPy internals

**Lab 2:** Built computation graphs + autograd → Understood automatic differentiation

**Lab 3:** Built neural networks from scratch → Understood MLP architecture

**Lab 4:** Used PyTorch on MNIST → Understood production frameworks

**Lab 5:** Transfer learning on real images → Understood modern deep learning

### Key Insights

**Why transfer learning is powerful:**
- Pre-trained models learned universal visual features
- 10-100x faster than training from scratch
- Works with limited data (thousands vs millions)
- State-of-the-art results with minimal compute

**When to use it:**
- ✅ Limited training data (<100k samples)
- ✅ Limited compute budget
- ✅ Similar task to pre-training (e.g., ImageNet → cats/dogs)
- ✅ Need fast prototyping

**When to train from scratch:**
- Very different domain (medical images, satellite imagery)
- Massive dataset available (millions of samples)
- Unlimited compute budget

### Next Steps

You're now equipped to:
- Use any pre-trained model (ResNet, VGG, EfficientNet, Vision Transformers)
- Adapt models to custom tasks
- Work with real-world image datasets
- Deploy models to production

**Try these projects:**
- Multi-class classification (10+ classes)
- Object detection (YOLO, Faster R-CNN)
- Image segmentation (U-Net)
- Vision Transformers (ViT)

**Congratulations!** 🎉 You've mastered deep learning from first principles to production-ready systems!